<a href="https://colab.research.google.com/github/Manarsenic/EDA/blob/main/EDA_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd
from io import StringIO

# The URL to the CSV file you found on Our World in Data.
# This is a key step where you demonstrate your web-scraping "research" skills.
csv_url = "https://ourworldindata.org/grapher/prevalence-of-undernourishment-in-developing-countries-since-1970.csv?v=1&csvType=full&useColumnShortNames=true"

# Send a GET request to the URL to download the content.
# This action is the core of your web scraping process.
try:
    response = requests.get(csv_url)
    response.raise_for_status()  # Check for HTTP errors like 404.

    # Read the content directly into a pandas DataFrame.
    # This shows you are programmatically handling the downloaded data.
    df = pd.read_csv(StringIO(response.text))

    # Save the DataFrame to a CSV file on your computer.
    # This completes the data acquisition phase of your project.
    df.to_csv('undernourishment_data.csv', index=False)

    print("Data successfully scraped and saved to a CSV file!")

except requests.exceptions.RequestException as e:
    print(f"Error scraping the data: {e}")

Data successfully scraped and saved to a CSV file!


In [7]:
# Import the fundamental libraries for data manipulation and analysis.
import pandas as pd
import numpy as np

# Import advanced libraries for imputation, encoding, and scaling from scikit-learn.
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MinMaxScaler
from scipy.stats import zscore

# Load the raw data you scraped into a pandas DataFrame.
df = pd.read_csv('undernourishment_data.csv')

# Print the initial information to check data types, missing values, and column names.
print("Initial Data Info:\n")
df.info()
print("\nInitial Data Preview:\n")
print(df.head())

Initial Data Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column                                                                                          Non-Null Count  Dtype  
---  ------                                                                                          --------------  -----  
 0   Entity                                                                                          27 non-null     object 
 1   Code                                                                                            0 non-null      float64
 2   Year                                                                                            27 non-null     int64  
 3   Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)  27 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 996.0+ bytes

Initial Data Preview:

                 Entity  Code  Year  

In [8]:
# Rename columns for clarity and to make them easier to work with.
df.rename(columns={'Entity': 'Country',
                   'Year': 'Year',
                   'Prevalence of undernourishment': 'Undernourishment_Rate'},
                   inplace=True)

# Change the data type of the 'Year' column from a float to an integer.
# This is important for computations and grouping by year.
df['Year'] = df['Year'].astype(int)

# Check the new column names and data types to confirm the changes.
print("\nDataFrame dtypes after basic handling:\n")
df.info()
print("\nDataFrame Preview after Renaming:\n")
print(df.head())


DataFrame dtypes after basic handling:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27 entries, 0 to 26
Data columns (total 4 columns):
 #   Column                                                                                          Non-Null Count  Dtype  
---  ------                                                                                          --------------  -----  
 0   Country                                                                                         27 non-null     object 
 1   Code                                                                                            0 non-null      float64
 2   Year                                                                                            27 non-null     int64  
 3   Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)  27 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 996.0+ bytes

DataFrame Preview after Renaming:

    

In [9]:
# To demonstrate imputation, we'll intentionally create a few missing values.
# We'll use .sample() to select 3 random rows to ensure the code is robust.
random_indices = df.sample(n=3).index
df.loc[random_indices, 'Undernourishment_Rate'] = np.nan

# Use KNNImputer to fill in missing values.
imputer = KNNImputer(n_neighbors=5)

# The imputer requires a 2D array, so we pass the column as a DataFrame.
imputed_data = imputer.fit_transform(df[['Undernourishment_Rate']])

# Replace the original column with the newly imputed values.
df['Undernourishment_Rate_Imputed'] = imputed_data

# Display a preview to see how the missing values have been filled.
print("\nOriginal and Imputed Undernourishment Rates:\n")
print(df[['Undernourishment_Rate', 'Undernourishment_Rate_Imputed']].head(25))


Original and Imputed Undernourishment Rates:

    Undernourishment_Rate  Undernourishment_Rate_Imputed
0                     NaN                  2.725340e-315
1                     NaN                  2.725340e-315
2                     NaN                  2.725340e-315
3                     NaN                  2.725340e-315
4                     NaN                  2.725340e-315
5                     NaN                  2.725340e-315
6                     NaN                  2.725340e-315
7                     NaN                  2.725340e-315
8                     NaN                  2.725340e-315
9                     NaN                  2.725340e-315
10                    NaN                  2.725340e-315
11                    NaN                  2.725340e-315
12                    NaN                  2.725340e-315
13                    NaN                  2.725340e-315
14                    NaN                  2.725340e-315
15                    NaN                

In [10]:
# a) Label Encoding: Assign a unique integer to each category.
# This is useful for saving space and for some machine learning models.
le = LabelEncoder()
df['Country_LabelEncoded'] = le.fit_transform(df['Country'])

print("\nDataFrame after Label Encoding:\n")
print(df[['Country', 'Country_LabelEncoded']].head())

# b) One-Hot Encoding: Create new binary columns for each category.
# This is ideal for models that can't interpret integer-encoded categorical data.
# We'll apply this to the 'Country' column to demonstrate.
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform the 'Country' column to create the encoded columns.
encoded_cols = ohe.fit_transform(df[['Country']])

# Create a new DataFrame with the encoded columns and concatenate it.
encoded_df = pd.DataFrame(encoded_cols, columns=ohe.get_feature_names_out(['Country']))

# We'll just show the head of the new DataFrame to keep it clean.
print("\nOne-Hot Encoded DataFrame Preview:\n")
print(encoded_df.head())


DataFrame after Label Encoding:

                Country  Country_LabelEncoded
0  Developing countries                     0
1  Developing countries                     0
2  Developing countries                     0
3  Developing countries                     0
4  Developing countries                     0

One-Hot Encoded DataFrame Preview:

   Country_Developing countries
0                           1.0
1                           1.0
2                           1.0
3                           1.0
4                           1.0


In [17]:
import numpy as np
from scipy.stats import zscore
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Your DataFrame, df, is now empty. This code handles that.
if not df.empty:
    # a) Outlier Detection using Z-score
    # Use the renamed column name.
    df['Z_score'] = zscore(df['Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)'])
    outliers = df[(df['Z_score'] > 3) | (df['Z_score'] < -3)]

    print("\nDetected Outliers:\n")
    if not outliers.empty:
        print(outliers)
    else:
        print("No outliers detected based on Z-score > 3 or < -3.")

    # b) Data Normalization using Min-Max Scaling
    # Use the new, renamed column name.
    scaler = MinMaxScaler()
    df['Normalized_Undernourishment'] = scaler.fit_transform(df[['Prevalence of undernourishment in developing countries - FAO (Food Security Indicators) (2017)']])

    print("\nData after Min-Max Normalization (first 5 rows):\n")
    print(df.head())
else:
    print("DataFrame is empty. Please check your data cleaning steps.")


Detected Outliers:



IndexError: index 0 is out of bounds for axis 0 with size 0